In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from functools import reduce

In [0]:
customers_df = spark.read.format("csv").option("inferSchema",True).option("header",True).load("/Workspace/Users/tekiyaswanthkumar@gmail.com/databricks_practice/pyspark_customers.csv")

In [0]:
transactions_df = spark.read.format("CSV").option("inferSchema",True).option("header",True).load("/Workspace/Users/tekiyaswanthkumar@gmail.com/databricks_practice/pyspark_transactions.csv")

In [0]:
transactions_df.printSchema()

In [0]:
customers_df.printSchema()

In [0]:
customers_df.count()

In [0]:
region_customers = customers_df.groupBy("region").agg(count('customer_id'))
active_customers = customers_df.groupBy("is_active").agg(count('customer_id'))
loyality_points_per_segment = customers_df.groupBy("segment").agg(avg('loyalty_points').alias("avg_loyalty_points"))
top_customers_cities = customers_df.groupBy("city").agg(count("customer_id").alias("count_of_customers")).orderBy(desc(col("count_of_customers"))).limit(5)


In [0]:
customers_df.filter(reduce(lambda x,y : x.isNull() | y.isNull(), [col(x) for x in customers_df.columns])).display()

In [0]:
transactions_df.limit(5).display()

In [0]:
# transactions_df.groupBy("status").agg(count('*').alias("count_of_transactions")).display()
# transactions_df.withColumn("net_price",round(col("gross_amount")*(1-col("discount_pct")), 3)).filter("status ='COMPLETED'").groupBy("product_category").agg(round(sum("net_price"),3).alias("net_revenue")).display()
transactions_df.filter("status = 'COMPLETED'").groupBy("payment_method").agg(round(avg("gross_amount"),2).alias("avg_gross")).orderBy(col("avg_gross").desc()).display()

In [0]:
customers_df.limit(2).display()
transactions_df.limit(2).display()

In [0]:
# cleansed_transactions_df = transactions_df.filter("status = 'COMPLETED'").dropDuplicates().dropna(subset=["customer_id"]).withColumn("net_price",col("gross_amount")*(1-col("discount_pct"))).groupBy("customer_id").agg(count('*').alias("total_transactions"), sum(col("quantity")).alias("total_quantity"), sum(col("gross_amount")).alias("total_gross_revenue"), sum("net_price").alias("total_net_revenue"), avg("net_price").alias("avg_transaction_value"))

from pyspark.sql.functions import *

cleansed_transactions_df = (
    transactions_df
    .filter("status = 'COMPLETED'")
    .dropDuplicates()
    .dropna(subset=["customer_id"])
    .withColumn("net_price", col("gross_amount") * (1 - col("discount_pct")))
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_transactions"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 3).alias("total_gross_revenue"),
        round(sum("net_price"), 3).alias("total_net_revenue"),
        round(avg("net_price"), 3).alias("avg_transaction_value")
    )
)


In [0]:
customers_df.join(cleansed_transactions_df, on = "customer_id", how = 'inner').select("customer_id", "customer_name","region", "total_transactions", "total_quantity", "total_gross_revenue", "total_net_revenue", "avg_transaction_value").orderBy(desc("total_net_revenue")).limit(10).display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

cdc_data = [
    (100001, "UPDATE", "Ravi Kumar", "Hyderabad", "ravi@gmail.com", "South", "Premium", 1200, "2026-11-13 09:00:00"),
    (100002, "UPDATE", "Suresh Reddy", "Chennai", "suresh.new@gmail.com", "South", "Standard", 800, "2026-11-13 09:05:00"),
    (100003, "UPDATE", "Priya Sharma", "Bangalore", "priya@gmail.com", "South", "Premium", 1500, "2026-11-13 09:10:00"),
    (100004, "INSERT", "Kiran Kumar", "Visakhapatnam", "kiran@gmail.com", "South", "Standard", 500, "2026-11-13 09:15:00"),
    (100005, "UPDATE", "Arjun Rao", "Pune", "arjun@gmail.com", "West", "Premium", 2500, "2026-11-13 09:20:00")
]

columns = [
    "customer_id", "operation", "name", "city", "email",
    "region", "segment", "loyalty_points", "updated_at"
]

cdc_source_df = spark.createDataFrame(cdc_data, columns)

cdc_source_df = cdc_source_df.withColumn(
    "updated_at",
    to_timestamp("updated_at")
)

display(cdc_source_df)

In [0]:
from delta.tables import DeltaTable

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customers")

In [0]:
%sql
describe extended dim_customers

In [0]:
dt = DeltaTable.forName(spark, "dim_customers")
dt.history().display()